In [ ]:
import torch

from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, IterableDataset

import json
from pyarrow import parquet as pq

import numpy as np

In [69]:
class LoadLanguageData(IterableDataset):
    
    def __init__(self, path:str, columns:list) -> None:

        super().__init__()

        self.cols = columns
        self.path = path
        self.dataset = pq.ParquetDataset(self.path)

    def __iter__(self):

        for fragment in self.dataset.fragments:

            table = fragment.to_table()
            rows = table.to_pylist()

            for row in rows:

                yield {
                    'input_ids' : torch.tensor(row[self.cols[0]], dtype=torch.long),
                    'output_ids': torch.tensor(row[self.cols[1]], dtype=torch.long),
                    'target_ids': torch.tensor(row[self.cols[2]], dtype=torch.long)
                }

def wrapper_collate_batch(pad_value):

    def collate_batch(batch):

        input_list  = [item['input_ids']  for item in batch]
        output_list = [item['output_ids'] for item in batch]
        target_list = [item['target_ids'] for item in batch]

        input_padded  = pad_sequence(input_list,  batch_first=True, padding_value=pad_value)
        output_padded = pad_sequence(output_list, batch_first=True, padding_value=pad_value)
        target_padded = pad_sequence(target_list, batch_first=True, padding_value=pad_value)

        return (input_padded, output_padded, target_padded)
    
    return collate_batch

In [70]:
with open("../data/main/vocabs/eng_tokenizer.json", "r") as f:
    tokenizer_eng = json.load(f)

with open("../data/main/vocabs/spa_tokenizer.json", "r") as f:
    tokenizer_spa = json.load(f)

In [71]:
word2index_eng = tokenizer_eng["vocab"]
index2word_eng = {int(key):value for key, value in tokenizer_eng["inverse_vocab"].items()}

word2index_spa = tokenizer_spa["vocab"]
index2word_spa = {int(key):value for key, value in tokenizer_spa["inverse_vocab"].items()}

In [72]:
train_spa_path = "../data/main/spa-eng/train_spa_eng"
train_spa_schema = pq.ParquetDataset(train_spa_path)
train_spa_cols = train_spa_schema.schema.names

train_spa_data = LoadLanguageData(train_spa_path, train_spa_cols)
train_spa_loader = DataLoader(train_spa_data, batch_size=64, collate_fn=wrapper_collate_batch(0))

In [73]:
for eng_in, spa_out, spa_t in train_spa_loader:
    break

print(eng_in.shape)
print(spa_out.shape)
print(spa_t.shape)

torch.Size([64, 14])
torch.Size([64, 11])
torch.Size([64, 11])


In [ ]:
max_key = max(index2word_eng.keys())
names_array = np.array(["Unknown"] * (max_key + 1), dtype=object)

for idx, val in index2word_eng.items():
    names_array[idx] = val

# 2. Map the values using vectorized indexing
# This returns a NumPy array of shape (batch, features) containing strings
mapped_output = names_array[eng_in.numpy()]

print(mapped_output)

[['i' 'do' "n't" 'have' 'the' 'address' 'with' 'me' '.' '[END]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]']
 ['i' 'do' "n't" 'have' 'the' 'address' 'now' '.' '[END]' '[PAD]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]']
 ['i' 'do' "n't" 'have' 'the' 'slightest' 'idea' 'what' 'it' "'s" 'about'
  '.' '[END]' '[PAD]']
 ['i' 'do' "n't" 'have' 'a' 'good' 'job' '.' '[END]' '[PAD]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]']
 ['i' 'do' "n't" 'have' 'a' 'car' '.' '[END]' '[PAD]' '[PAD]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]']
 ['i' 'do' "n't" 'have' 'a' 'lot' 'of' 'money' '.' '[END]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]']
 ['i' 'do' "n't" 'have' 'a' 'favorite' 'song' '.' '[END]' '[PAD]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]']
 ['i' 'do' "n't" 'have' 'a' 'favorite' 'shirt' '.' '[END]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]' '[PAD]']
 ['i' 'do' "n't" 'have' 'a' 'choice' '.' '[END]' '[PAD]' '[PAD]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]']
 ['i' 'do' "n't" 'have' 'a' 'ticket' '.' '[END]' '[PAD]' '[PAD]' '[PAD]'
  '[PAD]' '[PAD]' '[PAD]']
 ['i' 'do' "n